Develop and experiment with LLM prompts. Test whether the model correctly extracts event titles, dates, times, and other details from natural-language requests. Compare prompts and inspect mistakes.

In [ ]:
# Find your OpenAI API KEY
import importlib.metadata
import getpass
import io
import os
import sys
from pathlib import Path
import re
import warnings



class SetupError(Exception):
    """A setup problem with a plain-language fix. Notebooks show only the message, without a traceback."""

    def _render_traceback_(self):
        return [f"SetupError: {self}"]

def how_to_add_a_key():
    if IN_NOTEBOOK:
        return """No OpenAI API key found. Do one of these, then run this cell again:
- Colab: click the key icon in the left sidebar, add a secret named OPENAI_API_KEY, and turn on Notebook access.
- VS Code, Jupyter, or Anaconda: create a file named .env next to this notebook with one line:
  OPENAI_API_KEY=sk-...your key...
- Anywhere: set the OPENAI_API_KEY environment variable, then restart Jupyter."""
    return f"""No OpenAI API key found. Do one of these, then start the app again:
- Create a file named .env next to {Path(sys.argv[0]).name} (or in the folder you start it from) with one line:
  OPENAI_API_KEY=sk-...your key...
- Or set the OPENAI_API_KEY environment variable in this terminal."""

def find_env_file():
    """The nearest .env file: next to this script, in the working folder, or in a parent folder."""
    folders = [Path.cwd(), *Path.cwd().parents]
    if "__file__" in globals():
        folders.insert(0, Path(__file__).resolve().parent)
    for folder in folders:
        if (folder / ".env").is_file():
            return folder / ".env"
    return None


def where_is(path):
    """Where a file is, relative to the working folder. (A full path is long, and it can be personal.)"""
    folder, cwd = path.parent, Path.cwd()
    if folder == cwd:
        return f"the {path.name} file in this folder"
    if folder in cwd.parents:
        up = len(cwd.parts) - len(folder.parts)
        return f"the {path.name} file {up} folder{'s' if up > 1 else ''} up ({'../' * up}{path.name})"
    return f"the {path.name} file next to {Path(sys.argv[0]).name}"


def read_env_file(path):
    """KEY=VALUE pairs from a .env file (python-dotenv if it works, otherwise a small built-in parser).

    Also reads UTF-16 files, which Windows PowerShell 5 writes for `echo OPENAI_API_KEY=... > .env`.
    """
    data = path.read_bytes()
    utf16 = data[:2] in (b"\xff\xfe", b"\xfe\xff")
    if not utf16 and b"\x00" in data:  # UTF-16 without a BOM, e.g. PowerShell 5 `echo ... >> .env` added to a UTF-8 file
        data = data.replace(b"\x00", b"")  # the key and names are ASCII, so dropping the zero bytes recovers them
    try:
        text = data.decode("utf-16") if utf16 else data.decode("utf-8-sig")
    except UnicodeDecodeError:
        raise SetupError(
            f"{where_is(path)[0].upper() + where_is(path)[1:]} is not plain text (UTF-8). "
            f"Re-create it in VS Code or Notepad, save it as UTF-8, then {AGAIN}."
        ) from None
    try:
        from dotenv import dotenv_values

        return {k: v for k, v in dotenv_values(stream=io.StringIO(text)).items() if v is not None}
    except Exception:  # python-dotenv is missing or too old: parse the usual cases here
        pass
    values = {}
    for line in text.splitlines():
        line = re.sub(r"^export\s+", "", line.strip())
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = (part.strip() for part in line.split("=", 1))
        if value[:1] in ("'", '"') and value[0] in value[1:]:  # quoted: keep what is inside the quotes
            value = value[1 : value.index(value[0], 1)]
        else:
            value = re.split(r"\s+#", value, maxsplit=1)[0].strip()  # drop an inline comment
        values[name] = value
    return values


def key_from_colab():
    """The OPENAI_API_KEY secret from Colab Secrets, or None if it is not there."""
    from google.colab import userdata

    try:
        return userdata.get("OPENAI_API_KEY")
    except Exception as e:  # SecretNotFoundError or NotebookAccessError
        if type(e).__name__ == "NotebookAccessError":
            raise SetupError(
                "Colab has an OPENAI_API_KEY secret, but this notebook may not read it. Click the key icon "
                "in the left sidebar, turn on Notebook access for OPENAI_API_KEY, and run this cell again."
            ) from None
        return None


def can_prompt():
    """True if we can ask for the key: in a notebook, or in a script run from a terminal."""
    try:
        return IN_NOTEBOOK or sys.stdin.isatty()
    except (AttributeError, ValueError):  # no stdin at all, or it is closed
        return False


def load_api_key():
    """Find OPENAI_API_KEY and put it in os.environ. Returns where it came from; never prints the key.

    Order: environment variable, .env file, Colab Secrets, and finally a hidden prompt.
    """
    if os.environ.get("DEMO_RESTART_NEEDED"):  # the first cell asked for a restart; imports below would fail oddly
        raise SetupError(f"RESTART NEEDED: the first cell updated {os.environ['DEMO_RESTART_NEEDED']}, so {RESTART}.")
    source = None
    if os.environ.get("OPENAI_API_KEY", "").strip():
        source = "the OPENAI_API_KEY environment variable"
    else:
        os.environ.pop("OPENAI_API_KEY", None)  # an empty variable should not block the .env file
    env_file = find_env_file()
    if env_file:  # also loads other settings (for example RUN_PAID_EXTRAS), never overriding existing ones
        for name, value in read_env_file(env_file).items():
            if "\x00" not in name + value:  # os.environ cannot hold zero bytes
                os.environ.setdefault(name, value)
        if source is None and os.environ.get("OPENAI_API_KEY", "").strip():
            source = where_is(env_file)
    if source is None and IN_COLAB:
        key = key_from_colab()
        if key:
            os.environ["OPENAI_API_KEY"] = key
            source = "Colab Secrets"
    if source is None and can_prompt():
        try:
            key = getpass.getpass("Paste your OpenAI API key (hidden), then press Enter: ")
        except Exception:  # no way to ask, for example when the notebook runs unattended
            key = ""
        if key.strip():
            os.environ["OPENAI_API_KEY"] = key
            source = "the key you pasted (kept for this session only)"
    if source is None and env_file:
        raise SetupError(
            f"Found {where_is(env_file)}, but it has no OPENAI_API_KEY value. "
            f"Add the line OPENAI_API_KEY=sk-...your key... to it, then {AGAIN}."
        )
    if source is None:
        hint = "\nNote: this folder has a file named .env.txt; rename it to .env." if Path(".env.txt").exists() else ""
        raise SetupError(how_to_add_a_key() + hint)
    key = os.environ["OPENAI_API_KEY"].strip().strip("'\"“”‘’").replace("​", "").replace("﻿", "")
    if key.startswith("sk-your") or key.endswith("..."):
        raise SetupError(f"The key in {source} is still the placeholder text. Replace it with your real key.")
    if not key.isascii() or any(c.isspace() for c in key):
        raise SetupError(
            f"The key in {source} contains characters that are not part of an API key (for example curly quotes, "
            f"a space, or an invisible character from a word processor or chat app). Paste it again as plain text, then {AGAIN}."
        )
    os.environ["OPENAI_API_KEY"] = key
    return source

print(f"OpenAI API key loaded from {load_api_key()}.")

OpenAI API key loaded from the OPENAI_API_KEY environment variable.


The client and the five models this demo uses.
Each model name is written once, as a constant, so switching a model is a one-line change.

| Constant | Model | Used for | Price |
| --- | --- | --- | --- |
| `MODEL` | `gpt-6-luna` | Everything that generates text: chat, tools, agents, guardrail checks, the judge, images | $0.10 per 1M input tokens, $0.50 per 1M output tokens |
| `EMBED_MODEL` | `text-embedding-3-small` | Turning text into vectors for retrieval (Part 2) | $0.02 per 1M tokens |
| `MODERATION_MODEL` | `omni-moderation-latest` | Flagging harmful text (Part 2) | Free |
| `TTS_MODEL` | `gpt-4o-mini-tts` | Text to speech (section 5) | About $0.015 per minute of audio |
| `STT_MODEL` | `gpt-transcribe` | Speech to text (section 5) | $0.0045 per minute of audio |

In [ ]:
import openai
from openai import OpenAI

# The only models this demo uses. A course key may allow exactly these five and nothing else.
MODEL = "gpt-6-luna"  # every generation call: chat, tools, agents, guardrail checks, the judge, images
EMBED_MODEL = "text-embedding-3-small"  # turns text into vectors for retrieval
MODERATION_MODEL = "omni-moderation-latest"  # flags harmful text; free
TTS_MODEL = "gpt-4o-mini-tts"  # text to speech
STT_MODEL = "gpt-transcribe"  # speech to text
VOICE = "sage"  # one of the built-in text-to-speech voices

# Always set the reasoning effort: the model's default ("medium") is slow for simple tasks.
FAST_REASONING = {"effort": "none"}

client = OpenAI(timeout=60.0, max_retries=2)  # reads OPENAI_API_KEY; retries 429s, 5xx errors, and timeouts

Every call in these notebooks is logged: model, tokens, seconds of audio, latency, and estimated cost.
Your project needs the same kind of log, so feel free to copy this cell.

In [ ]:
# @title Usage log: tokens, audio, latency, and cost of every call (expand to read)
import time

import pandas as pd

# USD per 1M tokens: (input, cached input, output), standard tier. Update these if you change a model above.
# Prices change, so check https://developers.openai.com/api/docs/pricing
PRICES = {
    MODEL: (0.10, 0.01, 0.50),
    EMBED_MODEL: (0.02, 0.02, 0.0),
    MODERATION_MODEL: (0.0, 0.0, 0.0),  # free
}
# USD per minute of audio. Speech to text is billed per minute. Text to speech is billed per token
# ($0.60 per 1M text tokens in, $12 per 1M audio tokens out), which comes to about $0.015 per minute of speech.
AUDIO_PRICES = {TTS_MODEL: 0.015, STT_MODEL: 0.0045}
USAGE_LOG = []  # one row per model call


def log_usage(label, model, seconds, input_tokens=0, output_tokens=0, cached_tokens=0, reasoning_tokens=0, audio_seconds=0.0):
    """Add one model call to USAGE_LOG (tokens, audio, latency, estimated cost) and return the row."""
    known = [*PRICES, *AUDIO_PRICES]
    name = next((p for p in known if model == p or model.startswith(p + "-")), model)  # dated snapshots too
    if name not in known:
        print(f"Note: no price for {model!r}; add it to PRICES. Counting it as $0 for now.")
        PRICES[name] = (0.0, 0.0, 0.0)
    p_in, p_cached, p_out = PRICES.get(name, (0.0, 0.0, 0.0))
    cost = ((input_tokens - cached_tokens) * p_in + cached_tokens * p_cached + output_tokens * p_out) / 1e6
    cost += audio_seconds / 60 * AUDIO_PRICES.get(name, 0.0)
    row = {
        "label": label,
        "model": name,
        "input": input_tokens,
        "cached": cached_tokens,
        "output": output_tokens,
        "reasoning": reasoning_tokens,
        "audio_s": round(audio_seconds, 1),
        "seconds": round(seconds, 2),
        "cost_usd": cost,
    }
    USAGE_LOG.append(row)
    return row


def log_response(label, response, seconds):
    """Log a Responses API result: its usage has input, cached, output, and reasoning token counts."""
    u = response.usage
    cached = getattr(u.input_tokens_details, "cached_tokens", 0) or 0
    reasoning = getattr(u.output_tokens_details, "reasoning_tokens", 0) or 0
    return log_usage(label, response.model, seconds, u.input_tokens, u.output_tokens, cached, reasoning)


def usage_summary(rows=None):
    """Calls, tokens, audio seconds, summed latency, and cost per model, plus a total row."""
    df = pd.DataFrame(USAGE_LOG if rows is None else rows)
    if df.empty:
        return df
    summary = df.groupby("model").agg(
        calls=("label", "size"),
        input=("input", "sum"),
        cached=("cached", "sum"),
        output=("output", "sum"),
        reasoning=("reasoning", "sum"),
        audio_s=("audio_s", "sum"),
        seconds=("seconds", "sum"),
        cost_usd=("cost_usd", "sum"),
    )
    summary.loc["total"] = summary.sum()
    counts = ["calls", "input", "cached", "output", "reasoning"]
    return summary.astype({c: int for c in counts}).round({"audio_s": 1, "seconds": 1, "cost_usd": 5})

Test models general knowledge on calender events, such as holidays

In [ ]:
t0 = time.perf_counter()
response = client.responses.create(
    model=MODEL,
    # the rules: a role, and a format rule (say plainly what you want)
    instructions="You an assistant in charge of managing calender events. Do not use em dashes.",
    input="In two sentences, what holidays are coming up in November and December?",
    reasoning=FAST_REASONING,
    max_output_tokens=300,
)
row = log_response("first call", response, time.perf_counter() - t0)
print(response.output_text)

Test models knowledge given context of events

In [ ]:
from typing import Literal, get_args
from IPython.display import Markdown, display


def llm(label, model=MODEL, **kwargs):
    """Call the Responses API, log tokens, latency, and cost under `label`, and return the response.

    Sets the reasoning effort to "none" unless you pass one. With text_format=<Pydantic model> it uses
    responses.parse, which returns a structured output in response.output_parsed.
    """
    kwargs.setdefault("reasoning", FAST_REASONING)
    call = client.responses.parse if "text_format" in kwargs else client.responses.create
    t0 = time.perf_counter()
    response = call(model=model, **kwargs)
    log_response(label, response, time.perf_counter() - t0)
    return response


def show(text):
    """Display a model's answer as formatted Markdown (bold, lists) instead of raw asterisks."""
    display(Markdown(text))

EVENTS = [
    {
        "type" : "",
        "day" : "",
        "time" : "",
        "description" : ""
    }
]

response = llm(
    "with context",
    instructions = "",
    input = f""
)
print(response.output_text)